# CASCADE/OASIS Notebook for .mat traces

## Imports and Config

In [25]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.io import loadmat

sys.path.insert(0, os.path.join("libs", "CascadeTorch"))
sys.path.insert(0, os.path.join("libs", "OASIS"))

from cascade2p import cascade
from oasis.functions import deconvolve

traces_path = os.path.join(
    "libs",
    "CascadeTorch",
    "Example_datasets",
    "Multiplane-OGB1-zf-pDp-Rupprecht-7.5Hz",
    "Calcium_traces_04.mat",
)
ground_truth_path = os.path.join(
    "libs",
    "CascadeTorch",
    "Example_datasets",
    "Multiplane-OGB1-zf-pDp-Rupprecht-7.5Hz",
    "discrete_spikes_Calcium_traces_04.mat",
)
output_dir = os.path.join("data", "spike_detection_mat")
figures_dir = "figures"

frame_rate = 7.5
model_name = "Global_EXC_30Hz_smoothing25ms"
device_name = "cpu"
label = "Cascade .mat"

## Load dF/F traces

In [26]:
def load_mat_traces(path):
    mat = loadmat(path)
    return np.array(mat["dF_traces"])

def load_ground_truth_spike_times(path):
    mat = loadmat(path)
    return mat["spike_time_estimates"]

traces = load_mat_traces(traces_path).astype(np.float32, copy=False)
ground_truth_spike_times = load_ground_truth_spike_times(ground_truth_path)

valid_mask = np.isfinite(traces).all(axis=1)
valid_roi_indices = np.flatnonzero(valid_mask)
invalid_roi_count = (~valid_mask).sum()
traces = traces[valid_mask]
ground_truth_spike_times = [ground_truth_spike_times[0, roi_idx] for roi_idx in valid_roi_indices]

os.makedirs(output_dir, exist_ok=True)
np.save(os.path.join(output_dir, "input_dff_traces.npy"), traces)
np.save(os.path.join(output_dir, "valid_roi_indices.npy"), valid_roi_indices)

print("Trace path:", traces_path)
print("Ground truth path:", ground_truth_path)
print("Valid ROIs:", valid_roi_indices.size)
print("Invalid ROIs:", invalid_roi_count)
print("Trace shape:", traces.shape)
print("Ground truth entries:", len(ground_truth_spike_times))
print("dtype:", traces.dtype)

Trace path: libs/CascadeTorch/Example_datasets/Multiplane-OGB1-zf-pDp-Rupprecht-7.5Hz/Calcium_traces_04.mat
Ground truth path: libs/CascadeTorch/Example_datasets/Multiplane-OGB1-zf-pDp-Rupprecht-7.5Hz/discrete_spikes_Calcium_traces_04.mat
Valid ROIs: 1003
Invalid ROIs: 2
Trace shape: (1003, 260)
Ground truth entries: 1003
dtype: float32


## Run CASCADE

In [27]:
model_folder = os.path.join("libs", "CascadeTorch", "Pretrained_models")
device = torch.device(device_name)

cascade.download_model(model_name, model_folder=model_folder, verbose=1)

cascade_pred = cascade.predict(
    model_name,
    traces,
    model_folder=model_folder,
    device=device,
)

cascade_pred = cascade_pred.astype(np.float32)
np.save(os.path.join(output_dir, "cascade_spike_rates.npy"), cascade_pred)

print("cascade_pred shape:", cascade_pred.shape)

Pretrained model was saved in folder "/Users/arjunphanse/calcium-firing/libs/CascadeTorch/Pretrained_models/Global_EXC_30Hz_smoothing25ms"

 
The selected model was trained on 18 datasets, with 5 ensembles for each noise level, at a sampling rate of 30Hz, with a resampled ground truth that was smoothed with a Gaussian kernel of a standard deviation of 25 milliseconds. 
 

Loaded model was trained at frame rate 30 Hz
Given argument traces contains 1003 neurons and 260 frames.
Noise levels (mean, std; in standard units): 1.56, 0.42

Predictions for noise level 2:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 3:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 4:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 5:
	No neurons for this noise level

Predictions for noise level 6:
	No neurons for this noise leve

## Run OASIS

In [28]:
oasis_spikes = np.zeros_like(traces, dtype=np.float32)
oasis_calcium = np.zeros_like(traces, dtype=np.float32)

for neuron_idx, trace in enumerate(traces):
    calcium, spikes, _, _, _ = deconvolve(trace.astype(float), penalty=1)
    oasis_calcium[neuron_idx] = calcium
    oasis_spikes[neuron_idx] = spikes

np.save(os.path.join(output_dir, "oasis_spikes.npy"), oasis_spikes)
np.save(os.path.join(output_dir, "oasis_denoised_calcium.npy"), oasis_calcium)

print("oasis_spikes shape:", oasis_spikes.shape)

oasis_spikes shape: (1003, 260)


## Visualize dF/F, ground truth spikes, OASIS, CASCADE outputs for selected neurons

In [39]:
def normalize_for_plot(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    lo, hi = np.nanpercentile(values[finite], [1, 99])
    if hi <= lo:
        return np.zeros_like(values)
    return np.clip((values - lo) / (hi - lo), 0, 1)

noise = np.nanmedian(np.abs(np.diff(traces, axis=1)), axis=1) / 0.6745 / np.sqrt(2)
baseline = np.nanpercentile(traces, 50, axis=1)
peak = np.nanpercentile(traces, 99, axis=1)
roi_scores = (peak - baseline) / (noise + 1e-6)

roi_indices = np.argsort(roi_scores)[::-1][:5]

print("plot ROI indices:", roi_indices.tolist())

plot ROI indices: [961, 468, 924, 473, 70]


## Save comparison plots

In [40]:
os.makedirs(figures_dir, exist_ok=True)

n_neurons = len(roi_indices)
n_frames = min(traces.shape[1], round(frame_rate * 30))
time_axis = np.arange(n_frames) / frame_rate

fig, axes = plt.subplots(n_neurons, 1, figsize=(12, max(3, 1.8 * n_neurons)), sharex=True)
axes = np.atleast_1d(axes)
for plot_idx, ax in enumerate(axes):
    roi_idx = roi_indices[plot_idx]
    ax.plot(time_axis, traces[roi_idx, :n_frames], color="#555555", lw=1.0)
    ax.set_ylabel(f"ROI {roi_idx}")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Selected {label} dF/F traces")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "input_traces_mat.png"), dpi=200)
plt.close(fig)

fig, axes = plt.subplots(n_neurons, 1, figsize=(12, max(3, 1.8 * n_neurons)), sharex=True)
axes = np.atleast_1d(axes)
for plot_idx, ax in enumerate(axes):
    roi_idx = roi_indices[plot_idx]
    ax.plot(time_axis, normalize_for_plot(traces[roi_idx, :n_frames]), color="#777777", lw=0.9, zorder=3)
    spike_frames = np.asarray(ground_truth_spike_times[roi_idx]).ravel()
    spike_frames = spike_frames[np.isfinite(spike_frames)]
    spike_times_s = (spike_frames - 1) / frame_rate
    spike_times_s = spike_times_s[(spike_times_s >= 0) & (spike_times_s <= time_axis[-1])]
    ax.vlines(spike_times_s, 0.0, 1.05, color="#c44e52", lw=0.9, alpha=0.45, zorder=1)
    ax.plot(time_axis, normalize_for_plot(cascade_pred[roi_idx, :n_frames]), color="#2468a2", lw=1.1, zorder=4)
    ax.plot(time_axis, normalize_for_plot(oasis_spikes[roi_idx, :n_frames]), color="#19945f", lw=1.1, alpha=0.85, zorder=4)
    ax.set_ylabel(f"ROI {roi_idx}")
axes[-1].set_xlabel("Time (s)")
axes[0].legend(["dF/F", "Ground truth spikes", "CASCADE", "OASIS"], loc="upper right")
fig.suptitle(f"Ground truth spikes, CASCADE, and OASIS on selected {label} traces")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "cascade_oasis_traces_mat.png"), dpi=200)
plt.close(fig)